# Evolutionary Execution Researcher — LIVE mini-run (real LLM calls)

The companion to `execution_evolution_walkthrough.ipynb`: the same machinery, but
the **LLM-semantic mutation is real** — you see the exact prompt the mutating LLM
receives (parent code + the deterministic reflection brief + the executor
contract), its raw response, the validated child program, and the child's
deterministic evaluation.

* Needs `OPENAI_API_KEY` in `.env` (a few cents of the default research model).
* The market panel stays **synthetic** (AR(1), momentum-friendly) so the run is
  self-contained and finishes in ~a minute — the *live* part is the LLM, whose
  I/O is the thing worth inspecting here.
* The reward channel is byte-identical to the offline walkthrough: the LLM only
  ever *proposes code*; the deterministic harness scores it.


In [1]:
import json, os, sys, tempfile
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.insert(0, str(Path.cwd().parent))
from dotenv import load_dotenv
load_dotenv(Path.cwd().parent / ".env")
os.environ["QF_USE_MCP"] = "0"
assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY missing — set it in .env"

# synthetic AR(1) panel (see the walkthrough for why) + service wiring
N_BARS, TICKERS = 480, ["A", "B", "C", "D", "E", "F"]
rng = np.random.default_rng(3)
rets = np.zeros((N_BARS, len(TICKERS)))
eps = rng.standard_normal((N_BARS, len(TICKERS))) * 0.01
for t in range(1, N_BARS):
    rets[t] = 0.6 * rets[t - 1] + eps[t]
idx = pd.date_range("2024-01-01", periods=N_BARS, freq="D")
close = pd.DataFrame(100 * np.cumprod(1 + rets, axis=0), index=idx, columns=TICKERS)
PANEL = {"close": close}

from quant_fund_agent.mcp import research_service as svc
svc._load_panel_cached = lambda data_dir, fields, n_tickers: PANEL
svc._panel_cache_key = lambda data_dir, fields, n_tickers: ("live-panel",)

WORKDIR = Path(tempfile.mkdtemp(prefix="exec_live_"))
print("ready | workdir:", WORKDIR)


ready | workdir: /var/folders/0b/tfp69kqx3697_bkgxskvksmr0000gn/T/exec_live_sysd0i0w


## 1. Freeze the evaluation signals (v1) from a tiny factor book

In [2]:
FACTOR_TMPL = '''\
# Live-run factor {fid}

from __future__ import annotations

import pandas as pd

from quant_fund_agent.factors.base import BaseFactor
from quant_fund_agent.factors.ops import ts_rank
from quant_fund_agent.factors.registry import register_factor


@register_factor
class F_{fid}(BaseFactor):
    factor_id = "{fid}"
    name = "{fid}"
    category = "momentum"
    inputs = ["close"]
    prediction_horizon = 1

    def calc(self, data: dict[str, pd.DataFrame]) -> pd.DataFrame:
        close = data["close"]
        return {body}
'''
BOOK = [
    {"factor_id": "lv_mom",  "code": FACTOR_TMPL.format(fid="lv_mom",  body="close.pct_change().fillna(0.0)")},
    {"factor_id": "lv_rank", "code": FACTOR_TMPL.format(fid="lv_rank", body="(ts_rank(close, 8) - 0.5).fillna(0.0)")},
]
frozen = svc.freeze_signals(BOOK, out_dir=str(WORKDIR), version=1, target_horizon=1,
                            specs=[{"model": "ridge", "subset": [0, 1]},
                                   {"model": "ridge", "subset": [0]}])
assert frozen["ok"], frozen.get("error")
print("frozen K =", frozen["manifest"]["k"],
      "| poison audit passed:", frozen["manifest"]["poison_audit"]["passed"])
MANIFEST = frozen["manifest_path"]


frozen K = 2 | poison audit passed: True


## 2. One LLM mutation, step by step

Evaluate a seed → render its deterministic brief → build the mutation prompt →
**call the real LLM** → parse/validate/smoke-compile the child → evaluate the child
with the identical deterministic harness. Every intermediate is printed.


In [3]:
from quant_fund_agent.agents.execution_research.evolution.loop import _get_llm, _invoke
from quant_fund_agent.agents.execution_research.evolution.mutation import (
    build_exec_mutation_prompt, parse_exec_child_response,
)
from quant_fund_agent.agents.execution_research.evolution.reflection import exec_mutation_brief
from quant_fund_agent.agents.execution_research.evolution.seeds import seed_execution_programs
from quant_fund_agent.execution.codegen import compile_executor_inmem
from quant_fund_agent.mcp import research_client

seed = seed_execution_programs()[1]          # the per-underlying baseline
res = research_client.evaluate_executor_fitness(
    {"executor_id": seed.executor_id, "code": seed.code}, MANIFEST,
    n_trials=1, n_tickers=None)
assert res["ok"], res.get("error")
from quant_fund_agent.research_eval.fitness import FitnessResult
seed_fit = FitnessResult.from_dict(res["fitness"])
brief = exec_mutation_brief(seed_fit)
print("── the deterministic brief the LLM will see ──")
print(brief)


── the deterministic brief the LLM will see ──
EXECUTION FITNESS for `seed_zscore_threshold` (PASSED all gates)
- mean net VAL Sharpe (per bar) across 2 frozen signal(s): 0.608 (IS: 0.632, dispersion across signals: 0.003)
- cost efficiency (net/gross capture): 0.925 | mean turnover/bar: 0.357 | active on 0.895 of bars
- per-signal net VAL Sharpe: s0=0.605(capture 0.92), s1=0.611(capture 0.92)
- cost sensitivity (signal 0): half costs → 0.630, 1.5× costs → 0.580
ADVICE:
  1. All gates passed. Improve the dominated axes: raise net Sharpe via state-conditional sizing (vol targeting, drawdown de-risking), or cut cost drag while keeping the capture ratio.


In [4]:
prompt = build_exec_mutation_prompt(seed, brief, "live_child_1")
print(f"prompt: {len(prompt)} chars — head:\n")
print(prompt[:900], "…")


prompt: 4883 chars — head:

You are the Execution Researcher of a quant fund: you evolve the
PROGRAM that turns a strategy's composite alpha signal into a target book
through time (position sizing, entry/exit bands, risk overlays) — you never
touch the alpha itself.

PARENT PROGRAM (id: seed_zscore_threshold, regime: per_underlying):
mechanism: directional per-name band: act only on |z|>threshold extremes of each name's own history
```python
"""Baseline per-underlying executor: causal z-score, threshold band, 1/n."""

from __future__ import annotations

import numpy as np
import pandas as pd

from quant_fund_agent.execution.base import BaseExecutor, register_executor


@register_executor
class SeedZScoreThreshold(BaseExecutor):
    executor_id = "seed_zscore_threshold"
    name = "Seed: z-threshold equal-weight"
    regime = "per_underlying"
    inputs = ["signal"]
    params = {"n_max_positions": 6, "holding_perio …


In [5]:
llm = _get_llm(temperature=0.4, role="codegen")
raw = _invoke(llm, prompt)
print("── raw LLM response ──")
print(raw[:2200], "…" if len(raw) > 2200 else "")


── raw LLM response ──
{"executor_id": "live_child_1", "name": "Live Child: z-threshold with volatility scaling and drawdown de-risking", "regime": "per_underlying", "mechanism": "scales position sizes based on trailing volatility and reduces exposure during drawdowns", "expected_effect": "cuts turnover >=20% at <=10% capture loss", "code": "from __future__ import annotations\n\nimport numpy as np\nimport pandas as pd\n\nfrom quant_fund_agent.execution.base import BaseExecutor, register_executor\n\n@register_executor\nclass LiveChildZScoreThreshold(BaseExecutor):\n    executor_id = \"live_child_1\"\n    name = \"Live Child: z-threshold with volatility scaling and drawdown de-risking\"\n    regime = \"per_underlying\"\n    inputs = [\"signal\"]\n    params = {\"n_max_positions\": 6, \"holding_period\": 1, \"threshold\": 1.0, \"vol_scale_factor\": 0.5, \"drawdown_limit\": -0.1}\n\n    def target_weights(self, signal, state):\n        p = type(self).params\n        hp = int(p[\"holding_pe

In [6]:
payload = parse_exec_child_response(raw)
cls = compile_executor_inmem(payload["code"], "live_child_1")   # validator + smoke
print("child validated:", cls.__name__)
print("mechanism:      ", payload.get("mechanism"))
print("expected effect:", payload.get("expected_effect"))
print()
print(payload["code"])


child validated: LiveChildZScoreThreshold
mechanism:       scales position sizes based on trailing volatility and reduces exposure during drawdowns
expected effect: cuts turnover >=20% at <=10% capture loss

from __future__ import annotations

import numpy as np
import pandas as pd

from quant_fund_agent.execution.base import BaseExecutor, register_executor

@register_executor
class LiveChildZScoreThreshold(BaseExecutor):
    executor_id = "live_child_1"
    name = "Live Child: z-threshold with volatility scaling and drawdown de-risking"
    regime = "per_underlying"
    inputs = ["signal"]
    params = {"n_max_positions": 6, "holding_period": 1, "threshold": 1.0, "vol_scale_factor": 0.5, "drawdown_limit": -0.1}

    def target_weights(self, signal, state):
        p = type(self).params
        hp = int(p["holding_period"])
        if hp > 1:
            mask = np.arange(len(signal)) % hp == 0
            signal = signal.where(pd.Series(mask, index=signal.index), other=np.nan)
        

In [7]:
child_res = research_client.evaluate_executor_fitness(
    {"executor_id": "live_child_1", "code": payload["code"]}, MANIFEST,
    n_trials=2, n_tickers=None)
if child_res["ok"]:
    child_fit = FitnessResult.from_dict(child_res["fitness"])
    cmp = pd.DataFrame({"seed": seed_fit.objective.to_dict(),
                        "LLM child": child_fit.objective.to_dict()})
    print("gates (child):", json.dumps(child_fit.gates.to_dict(), indent=2))
    display(cmp)
else:
    print("child evaluation failed (a scored outcome, not a crash):",
          child_res.get("error"))


gates (child): {
  "coverage_ok": true,
  "degradation_ok": true,
  "deflation_ok": null,
  "cost_ok": true,
  "passed": true
}


,seed,LLM child
marginal_value,0.607994,0.458845
independence,0.606587,0.458503
robustness,0.924709,0.917521
parsimony,-41.000000,-54.000000
structural_novelty,NaN,NaN


## 3. The full loop, live (1 generation, LLM-semantic children)

The same `ExecEvolutionLoop` as the walkthrough, with `p_llm_semantic=0.7` — watch
the lineage record which operator produced each admitted child.


In [8]:
from quant_fund_agent.agents.execution_research.evolution.loop import (
    ExecEvolutionLoop, ExecEvolutionRunConfig,
)

cfg = ExecEvolutionRunConfig(
    out_dir=str(WORKDIR / "evolution_exec"), signals_manifest=MANIFEST,
    generations=1, population_size=6, children_per_generation=3,
    seed=11, n_tickers=None,
    p_llm_semantic=0.7, p_crossover=0.0, p_jitter=0.3)
loop = ExecEvolutionLoop(cfg)
summary = loop.run()

print(f"n_trials={summary['n_trials']}  archive={len(summary['archive'])}  "
      f"SOTA={summary['sota_executor']}  eval_failures={summary['n_eval_failures']}")
pd.DataFrame([
    {"gen": r["generation"], "op": r["operator"], "id": r["factor_ids"][0][:28],
     "selectable": r["selectable"],
     "net_sharpe": (r["objective"] or {}).get("marginal_value")}
    for r in loop.controller.lineage if "operator" in r])


n_trials=5  archive=3  SOTA=seed_topk_dollar_neutral  eval_failures=0


,gen,op,id,selectable,net_sharpe
0,0,seed,seed_topk_dollar_neutral,True,0.622026
1,0,seed,seed_zscore_threshold,True,0.607994
2,1,llm_semantic,ex1_1_9f69,True,0.615119
3,1,llm_semantic,ex1_2_1c21,True,0.615119
4,1,llm_semantic,ex1_3_263b,True,0.615119


That is the whole E0–E2 stack live: deterministic reward, LLM-only-proposes,
auditable lineage. The joint outer layer (J-phases) will drive this loop in blocks
against the factor arm — see `docs/joint-evolution/DESIGN.md`.
